In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

from bellm.tokeniser import Tokeniser
from reader import get_dataset

/Users/belle/Developer/Belllm/belllm/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
DEVICE = torch.device("mps")
EPOCHS = 50
BATCH_SIZE = 16

# Token window generation length
TOKEN_GENERATION_LENGTH = 200

In [3]:

class TextDiffusionTransformer(nn.Module):
    def __init__(self, vocab_size=10000, context_len=1000, diffuse_len=200, model_dim=512):
        super().__init__()

        self.diffuse_len = diffuse_len
        self.model_dim = model_dim

        # 1. Context Embedding (for the 1k fixed tokens)
        self.context_embedding = nn.Embedding(vocab_size, model_dim)

        # 2. Logit Projection (for the 200 tokens being diffused)
        # This takes the B x 200 x 10000 logits and brings them to model_dim
        self.diffuse_projection = nn.Linear(vocab_size, model_dim)

        # 3. Time Embedding (MLP)
        self.time_mlp = nn.Sequential(
            nn.Linear(1, model_dim),
            nn.SiLU(),
            nn.Linear(model_dim, model_dim)
        )

        # 4. Transformer Layers
        # We use a standard Encoder; you can increase num_layers for better results
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=model_dim,
            nhead=8,
            dim_feedforward=model_dim * 4,
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=6)

        # 5. Output Head (Back to 10k logits)
        self.to_velocity = nn.Linear(model_dim, vocab_size)

    def forward(self, context_ids, noisy_logits, t):
        """

        noisy_logits: (B, 200, 10000) - Continuous logit space
        t: (B, 1) - Timestep [0, 1]
        """
        # todo need positional embedding

        # Embed context
        c_emb = self.context_embedding(context_ids)  # (B, 1000, dim)

        # Project noisy logits
        d_emb = self.diffuse_projection(noisy_logits)  # (B, 200, dim)

        # Combine sequences (Total length: 1200)
        x = torch.cat([c_emb, d_emb], dim=1)

        # # Inject Time
        t_emb = self.time_mlp(t).unsqueeze(1)  # (B, 1, dim)
        x = x + t_emb

        # Process through Transformer
        x = self.transformer(x)

        # Slice out only the 200 "diffuse" tokens for the output
        diffuse_out = x[:, -self.diffuse_len:, :]

        pred_x0 = self.to_velocity(diffuse_out)
        # pred_x0 = F.softmax(pred_x0, dim=-1)

        # The velocity on the simplex is the straight line:
        # v = (pred_x0 - noise) / (1 - t)
        return pred_x0


In [5]:
vocab_size = 5001

model = TextDiffusionTransformer(vocab_size=vocab_size, context_len=800, diffuse_len=TOKEN_GENERATION_LENGTH)
model.to(DEVICE)
print("Model Size:", sum(p.numel() for p in model.parameters()))

dataset = get_dataset()
tokeniser = Tokeniser().load(f"../src/bellm/tokeniser/tokeniser.json")
for k in list(tokeniser.token_map):
    if tokeniser[k] > 5000:
        del tokeniser.token_map[k]

items = list(dataset \
     # .skip(i * SAMPLE_SIZE_BATCH_SIZE + start_offset) \
     .take(50_000)["text"])
print("Got items")

text_tokens = tokeniser.tokenize_batch(items)
print("Tokenised")

# todo create train / test pairs properly and create masking

subset = [
    item.token_ids[:600+TOKEN_GENERATION_LENGTH]
    for item in text_tokens
    if len(item.token_ids) > 800
]
print("Created subset")

X = torch.tensor(np.array([x[:600] for x in subset]), dtype=torch.long, device=DEVICE)
Y = torch.tensor(np.array([x[600:] for x in subset]), dtype=torch.long, device=DEVICE)


Model Size: 26865033


Got items
Tokenised
Created subset


In [6]:
optimiser = torch.optim.AdamW(model.parameters())
loss = nn.CrossEntropyLoss()

def tensor(x):
    return torch.tensor(x, dtype=torch.long, device=DEVICE)

for epoch in range(EPOCHS):
    model.train()
    losses = []

    for b_idx in range(0, len(X), BATCH_SIZE):
        batch_x = X[b_idx:b_idx + BATCH_SIZE]
        batch_y = Y[b_idx:b_idx + BATCH_SIZE]

        optimiser.zero_grad()
        noisy_input = torch.randn((len(batch_x), TOKEN_GENERATION_LENGTH, vocab_size), dtype=torch.float)

        # Create the outputs the model should be generating in perfect world
        x0 = F.one_hot(batch_y, num_classes=vocab_size).float() # (B, 200, 10000)

        # 2. Create the "Noise" state (x1)
        # For a distribution, noise is usually a Uniform Distribution (Maximum Entropy)
        x1 = torch.ones_like(x0) / vocab_size

        # 3. Sample a random time t for each batch [0, 1]
        t = torch.rand(len(batch_x), 1, device=DEVICE)
        t_expanded = t.unsqueeze(-1) # (B, 1, 1) to match (B, 200, 10000)

        # 4. Construct the noisy flow xt
        # Linear interpolation between the clean one-hot and the uniform noise
        xt = (1 - t_expanded) * x0 + t_expanded * x1

        # 5. Model predicts the "Clean Logits"
        # Even though we feed in xt (probabilities), the model outputs
        # the 10k logits for the target x0.
        pred_logits = model(batch_x, xt, t) # (B, 200, 10000)

        # 6. Categorical Cross-Entropy Loss
        # This forces the model to recover the discrete IDs from the noisy mixture
        loss = F.cross_entropy(
            pred_logits.view(-1, vocab_size),
            batch_y.view(-1)
        )

        loss.backward()
        optimiser.step()

        l = loss.item()
        losses.append(l)

        print(f"\rEpoch: {epoch}. Batch: {b_idx}/{len(X)}.", str(np.mean(losses)), end="")

    print(f"\rEpoch: {epoch}", str(np.mean(losses)))


Epoch: 0 7.81408399672020254. 7.8140839967202025
Epoch: 1 7.77793676646675654. 7.7779367664667565
Epoch: 2 7.77647924197940454. 7.7764792419794045
Epoch: 3. Batch: 8800/10154. 7.7723882964215565

KeyboardInterrupt: 

In [25]:
model.eval()

text = "4 + 4 = "
tokens = tokeniser.tokenize_batch([text])
model_input = torch.tensor(np.array([x.token_ids for x in tokens]), dtype=torch.long, device=DEVICE)
# x0 = torch.ones((1, TOKEN_GENERATION_LENGTH, vocab_size)) / vocab_size
# # x0.to(DEVICE)
# output = model(model_input, x0.to(DEVICE), torch.zeros((1, 1), device=DEVICE))
# print(torch.argmax(F.softmax(output, dim=1), dim=1).detach().cpu().numpy())
#
# output = model(model_input, x0.to(DEVICE), torch.ones((1, 1), device=DEVICE) * 0.25)
# print(torch.argmax(F.softmax(output, dim=1), dim=1).detach().cpu().numpy())
#
# output = model(model_input, x0.to(DEVICE), torch.ones((1, 1), device=DEVICE) * 0.5)
# print(torch.argmax(F.softmax(output, dim=1), dim=1).detach().cpu().numpy())
#
# output = model(model_input, x0.to(DEVICE), torch.ones((1, 1), device=DEVICE) * 0.75)
# print(torch.argmax(F.softmax(output, dim=1), dim=1).detach().cpu().numpy())

# output = model(model_input, output, torch.zeros((1, 1), device=DEVICE))



def generate_flow(model, context_ids, steps=20):
    B = context_ids.shape[0]
    # Start at t=1 (Pure Noise)
    xt = torch.randn(B, TOKEN_GENERATION_LENGTH, vocab_size).to(context_ids.device)

    dt = 1.0 / steps

    for i in range(steps):
        # Current time (1.0 down to 0.0)
        t_val = 1.0 - (i * dt)
        t = torch.full((B, 1), t_val).to(context_ids.device)

        # Predict velocity
        v = model(context_ids, xt, t)

        # Euler Step: Update xt by moving in the direction of -velocity
        # (Minus because we are going from noise -> data)
        xt = xt - v * dt

    return xt # These are your denoised pre-softmax logits

softmax = F.softmax(generate_flow(model, model_input), dim=2)
tokens = torch.argmax(softmax, dim=2).detach().cpu().numpy()

"".join(tokeniser.detokenise(tokens[0]))

'nication\n\n\n signific signific\nnication\n\n\nnication signific.mnute\nnication\nnication\n significnication signific\nnication significnicationnicationnicationnicationnicationnicationgori\nnicationnication signific significnicationnication\n\nnicationnutenication signific signific signific\ngori signific significnication significnication\nnication\n significnication\n\nnication signific signific\n signific\n significnicationnicationnicationnication significnicationnicationampa\nnicationnicationnicationnicationnicationgorinication\nnication\n significnication signific\nnicationnicationnicationnicationnicationnication’t \nnicationnicationnication signific signific\n significnicationnication significnication\nnicationnication significnicationnicationnication significnication\nnicationnication\nnicationnicationnication\nnicationnication significnication’t  signific signific signific\nnication\n\n\nnicationnication\n\n\n signific\n signific\nnicationnication\n\ngorinication\n\nnicationn

In [23]:
tokens

array([[4052,  852,   57, 4052,   57,   57, 4052,   57, 4052,   57, 4052,
        4233, 4052, 4052, 4052, 4052, 4233, 4052, 4052, 4233, 4052, 4052,
        4052, 4052, 2960,   57, 4052, 4052, 4583, 4052, 4233,  852,   57,
        4233, 4233, 4052, 4052, 4233, 4052,   57, 4052, 4052, 4052, 4052,
        4233, 4233,   57,   57, 4052, 4052, 4052, 4052, 4233, 4052, 4233,
        4052, 4233,   57,   57, 4052, 4233,   57,   57, 4233, 4583, 4052,
        4052, 4052, 4031, 4052,   57, 4052, 4052, 4052, 4052,   57, 4052,
        4052,   57, 4233,   57,  852, 4052,   57, 4052, 4052, 4052, 4233,
        4052,   57, 4052, 4052,   57, 4052, 4052, 4052, 4052, 4052, 4233,
          57,   57, 4052, 4052, 4052, 4052,   57, 4052,   57,   57, 4233,
        4233, 4052, 4052, 4052,   57, 4233,   57, 4233, 4052, 4052, 4052,
          57, 4233, 4233, 4233, 4052,   57, 4052,   57, 4233, 4233, 4233,
        4233, 4052, 4052, 4233, 4052, 4052, 4052, 4052,   57, 4052, 4052,
        4052, 4052, 4233, 4052,   57, 